# Column Transformer with Mixed Types

This compares native Moxie with scikit-learn's
[mixed-types gallery example](https://scikit-learn.org/dev/auto_examples/compose/plot_column_transformer_mixed_types.html)
at revision `4358542a1bd39af2e7e1cbace8b9fed067473c5b`.

`ColumnTransformer` dispatches Titanic's numeric columns to median
imputation and standard scaling. It sends categorical columns to dense
one-hot encoding and chi-squared feature selection. Logistic regression fits
the resulting dense matrix.

The notebook defaults to seeded Titanic-like data for offline runs. Set
`MOXIE_TITANIC_OPENML=1` to use the gallery's [OpenML Titanic dataset
(version 1)](https://www.openml.org/d/40945). The named route sends passenger
class to one-hot encoding. Moxie uses string labels for passenger class in
that route so the branch can encode all three columns in one categorical
domain. The dtype route keeps passenger class as Int64 and sends it to
imputation and scaling. The target is Float64 for chi-squared feature
selection and Int64 for classification.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("compose/plot_column_transformer_mixed_types")

## scikit-learn reference

In [2]:
import os

import numpy as np
import pandas as pd

use_openml = os.environ.get("MOXIE_TITANIC_OPENML") == "1"
if use_openml:
    from sklearn.datasets import fetch_openml

    X, y = fetch_openml(
        data_id=40945, data_home=ROOT / "builds/titanic-reference",
        as_frame=True, return_X_y=True,
    )
    assert X.shape == (1309, 13)
    X = X[["embarked", "sex", "pclass", "age", "fare"]].copy()
    y = y.astype(np.int64)
else:
    rng = np.random.RandomState(0)
    n = 1309
    pclass = rng.choice([1, 2, 3], size=n, p=[0.25, 0.2, 0.55])
    sex = rng.choice(["female", "male"], size=n, p=[0.36, 0.64])
    embarked = rng.choice(["C", "Q", "S"], size=n, p=[0.2, 0.1, 0.7])
    age = np.round(rng.normal(30 - 3 * (pclass - 2), 13, size=n).clip(0.5, 80), 1)
    fare = np.round(rng.lognormal(3.6 - 0.6 * (pclass - 1), 0.6, size=n), 2)
    age[rng.rand(n) < 0.2] = np.nan
    fare[rng.rand(n) < 0.01] = np.nan
    logit = 1.2 * (sex == "female") - 0.8 * (pclass - 2) - 0.02 * np.nan_to_num(age - 30) + 0.3 * (embarked == "C") - 0.6
    survived = (rng.rand(n) < 1 / (1 + np.exp(-2 * logit))).astype(int)
    X = pd.DataFrame({"embarked": pd.Categorical(embarked), "sex": pd.Categorical(sex),
                      "pclass": pclass, "age": age, "fare": fare})
    y = pd.Series(survived, name="survived")
if use_openml:
    print("OpenML Titanic (40945)")
X.head()

,embarked,sex,pclass,age,fare
0,S,male,3,13.8,18.66
1,S,male,3,27.6,18.32
2,C,female,3,NaN,15.93
3,S,male,3,17.4,9.90
4,S,female,2,54.7,25.02


In [3]:
import runpy

import sklearn.compose
import sklearn.feature_selection
import sklearn.impute
from sklearn.model_selection import train_test_split
from compose_parity_io import load_pinned, pinned_code, verify_reference

# Use the locked sklearn wheel for supporting imports and the pinned source
# checkout for the code under comparison.
sources = {
    "compose": "sklearn/compose/_column_transformer.py",
    "pipeline": "sklearn/pipeline.py",
    "impute": "sklearn/impute/_base.py",
    "scaler": "sklearn/preprocessing/_data.py",
    "encoders": "sklearn/preprocessing/_encoders.py",
    "encode": "sklearn/utils/_encode.py",
    "selection": "sklearn/feature_selection/_univariate_selection.py",
    "example": "examples/compose/plot_column_transformer_mixed_types.py",
}
verify_reference(sources.values())
load_pinned(sources["encode"], register=True)
pipeline = load_pinned(sources["pipeline"], register=True)
compose = load_pinned(sources["compose"])
impute = load_pinned(sources["impute"])
scaler = load_pinned(sources["scaler"])
encoders = load_pinned(sources["encoders"])
selection = load_pinned(sources["selection"])
for method, key in [
    (compose.ColumnTransformer.fit_transform, "compose"),
    (pipeline.Pipeline.fit_transform, "pipeline"),
    (impute.SimpleImputer.fit, "impute"),
    (scaler.StandardScaler.fit, "scaler"),
    (encoders.OneHotEncoder.fit, "encoders"),
    (selection.chi2, "selection"),
    (selection.SelectPercentile._get_support_mask, "selection"),
]:
    pinned_code(method, sources[key])

logistic_oracle = runpy.run_path(str(
    ROOT / "families/linear_model/algorithms/logistic/tests/parity.py"
))
LogisticRegression = logistic_oracle["oracle"]()

numeric = ["age", "fare"]
categorical = ["embarked", "sex", "pclass"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)
# Exercise named routing when transform receives a different column order.
X_test = X_test[numeric + categorical[::-1]]
non_category_selector = compose.make_column_selector(dtype_exclude="category")
category_selector = compose.make_column_selector(dtype_include="category")


def preprocessor(num_columns, cat_columns):
    return compose.ColumnTransformer([
        ("num", pipeline.Pipeline([
            ("imputer", impute.SimpleImputer(strategy="median")),
            ("scaler", scaler.StandardScaler()),
        ]), num_columns),
        ("cat", pipeline.Pipeline([
            ("encoder", encoders.OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ("selector", selection.SelectPercentile(selection.chi2, percentile=50)),
        ]), cat_columns),
    ])


references = {
    "named": preprocessor(numeric, categorical),
    "dtype": preprocessor(non_category_selector, category_selector),
}
assert non_category_selector(X_train) == ["pclass", "age", "fare"]
assert category_selector(X_train) == ["embarked", "sex"]

nb.share(y_train=y_train, y_test=y_test)
for split, frame in [("train", X_train), ("test", X_test)]:
    nb.share(**{f"{split}_numeric": frame[numeric].to_numpy(),
                f"{split}_pclass_integer": frame["pclass"].to_numpy(dtype=np.int64)})
    for name in categorical:
        labels = frame[name].astype(object)
        nb.share(**{f"{split}_{name}": labels.fillna("").to_numpy(),
                    f"{split}_{name}_missing": labels.isna().to_numpy()})

scikit-learn source SHA256 {'sklearn/linear_model/_linear_loss.py': 'db6fd7e23b4619a76a46b8225876646328930db42d42e6af76223f72674afa3e', 'sklearn/linear_model/_logistic.py': '30863a8c7be82a63e2994405cf3fb29a4ab069eccb74ea6720582b4d8dec0b99', 'sklearn/linear_model/_glm/_newton_solver.py': 'f9c7caf323700d69d52ff709ea5e82d5c5fcfaa433f216cadaffce81621f6eae', 'sklearn/utils/optimize.py': '5f8d9a9ca0fb8e15624f4669f9ed5a2afe74c4e119cc4cb392a16df4eaeee575', 'sklearn/svm/_base.py': 'be4d5a830d357be28e2a85d6c80c208d8aa8536858a62e17cad50abf0f7fe62f', 'sklearn/svm/_liblinear.pyx': '91f175ac10f6d6b711fb09f975f2d5aa1cd9416bcc1cc668ae5be6eb250a7480', 'sklearn/svm/src/liblinear/linear.cpp': 'c5329418a6de7470ead7a858fd57b6e686ada53d102235cb3e1457fd0137a9d4', 'sklearn/svm/src/liblinear/linear.h': 'c3bd0dfc7bbc35a4ec9b16137df5df3938296cad676dca73540893d4e3ac88db', 'sklearn/svm/src/liblinear/tron.cpp': '4b942fe335f1475d89048c2b3492d14c5884975b9beeae3604bba5605076400d', 'sklearn/svm/src/liblinear/tron.h': '

## Moxie

Each branch is a native transformer pipeline. `ColumnTable` owns the numeric
matrix and typed categorical columns. The notebook passes the transformed
matrices between Mojo cells to keep compilation within the memory cap.

In [4]:
%%mojo mixed_preprocessor
from moxie.compose import ColumnEntry, Columns, ColumnTransformer, make_column_selector
from moxie.feature_selection import BuiltinScorer, SelectPercentile
from moxie.impute import SimpleImputer
from moxie.pipeline import make_pipeline
from moxie.preprocessing import OneHotEncoder, StandardScaler
from moxie_nb_io import load, save, save_ints, save_strings
from titanic_table import load_table


def run_route(route: String, named: Bool) raises:
    var train = load_table("train", named)
    var test = load_table("test", named)
    var numeric: Columns[]
    var categorical: Columns[]
    if named:
        numeric = Columns(["age", "fare"])
        categorical = Columns(["embarked", "sex", "pclass"])
    else:
        var category: List[String] = ["category"]
        var num_selector = make_column_selector(dtype_exclude=category.copy())
        var cat_selector = make_column_selector(dtype_include=category^)
        save_strings("dtype_numeric", num_selector(train).names.copy())
        save_strings("dtype_categorical", cat_selector(train).names.copy())
        save_strings("dtype_schema", train.schema().dtypes.copy())
        numeric = Columns(num_selector^)
        categorical = Columns(cat_selector^)

    var num = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    var cat = make_pipeline(
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        SelectPercentile(BuiltinScorer("chi2"), percentile=50),
    )
    var preprocessor = ColumnTransformer(
        ColumnEntry("num", num^, numeric^),
        ColumnEntry("cat", cat^, categorical^),
    )
    save(route + "_train", preprocessor.fit_transform(train, load("y_train")))
    save(route + "_test", preprocessor.transform(test))
    save_strings(route + "_names", preprocessor.get_feature_names_out())
    var spans = List[Int]()
    if preprocessor.output_indices_:
        ref indices = preprocessor.output_indices_.value()
        for name in ["num", "cat", "remainder"]:
            spans.append(indices[name].start)
            spans.append(indices[name].stop)
    save_ints(route + "_spans", spans)


def main() raises:
    run_route("named", True)
    run_route("dtype", False)

In [5]:
mx = nb.results["mixed_preprocessor"]
nb.share(**{f"MX_{route}_{split}": mx[f"{route}_{split}"]
            for route in references for split in ("train", "test")})

In [6]:
%%mojo logistic_regression
from moxie.linear_model import LogisticRegression
from moxie_nb_io import load, save, save_floats


def main() raises:
    var y_train = load[DType.int64]("y_train")
    var y_test = load[DType.int64]("y_test")
    for route in ["named", "dtype"]:
        var train = load("MX_" + route + "_train")
        var test = load("MX_" + route + "_test")
        var model = LogisticRegression(tol=1e-9)
        _ = model.fit(train, y_train)
        save(route + "_pred", model.predict(test))
        save(route + "_proba", model.predict_proba(test))
        save_floats(route + "_accuracy", [model.score(test, y_test)])

## Parity

In [7]:
nb.compare("Dtype route numeric selector", non_category_selector(X_train),
           mx["dtype_numeric"], exact=True)
nb.compare("Dtype route categorical selector", category_selector(X_train),
           mx["dtype_categorical"], exact=True)
nb.compare("Dtype route input schema", X_train.dtypes.astype(str).to_numpy(),
           mx["dtype_schema"], exact=True)

classified = nb.results["logistic_regression"]
for route, reference in references.items():
    sk_train = reference.fit_transform(X_train, y_train)
    sk_test = reference.transform(X_test)
    sk_names = reference.get_feature_names_out()
    print(f"{route.capitalize()} route")
    moxie_nb.side_by_side(sklearn=pd.DataFrame(sk_test[:5], columns=sk_names),
                          moxie=pd.DataFrame(mx[f"{route}_test"][:5],
                                             columns=mx[f"{route}_names"]))
    sk_spans = [edge for name in ["num", "cat", "remainder"]
                for edge in (reference.output_indices_[name].start,
                             reference.output_indices_[name].stop)]
    nb.compare(f"{route} feature names", sk_names, mx[f"{route}_names"], exact=True)
    nb.compare(f"{route} output_indices_", sk_spans, mx[f"{route}_spans"], exact=True)
    num_span = reference.output_indices_["num"]
    cat_span = reference.output_indices_["cat"]
    for split, expected in [("train", sk_train), ("test", sk_test)]:
        actual = mx[f"{route}_{split}"]
        nb.compare(f"{route} {split} numeric output", expected[:, num_span], actual[:, num_span],
                   rtol=1e-12, atol=1e-12,
                   reason="Float64 scaler reductions can sum in different orders.")
        nb.compare(f"{route} {split} selected one-hot output", expected[:, cat_span],
                   actual[:, cat_span], exact=True)

    sk_classifier = LogisticRegression(tol=1e-9).fit(sk_train, y_train)
    nb.compare(f"{route} classifier predictions", sk_classifier.predict(sk_test),
               classified[f"{route}_pred"], exact=True)
    # The seeded offline dtype fit differs by about 6.4e-6 in probability.
    proba_atol = 7e-6 if route == "dtype" and not use_openml else 1e-8
    nb.compare(f"{route} classifier probabilities", sk_classifier.predict_proba(sk_test),
               classified[f"{route}_proba"], rtol=1e-6, atol=proba_atol,
               reason="L-BFGS line search and convergence can differ between implementations.")
    nb.compare(f"{route} classifier accuracy", [sk_classifier.score(sk_test, y_test)],
               classified[f"{route}_accuracy"], exact=True)


Named route


,num__age,num__fare,cat__sex_female,cat__sex_male,cat__pclass_1,cat__pclass_3
0,1.596599,-0.835759,0.0,1.0,0.0,1.0
1,-0.107641,-0.451979,1.0,0.0,0.0,0.0
2,0.349594,-0.258225,0.0,1.0,0.0,1.0
3,-0.440176,0.579664,0.0,1.0,1.0,0.0
4,-0.091014,0.551719,0.0,1.0,0.0,1.0
,num__age,num__fare,cat__sex_female,cat__sex_male,cat__pclass_1,cat__pclass_3
0,1.596599,-0.835759,0.0,1.0,0.0,1.0
1,-0.107641,-0.451979,1.0,0.0,0.0,0.0
2,0.349594,-0.258225,0.0,1.0,0.0,1.0
3,-0.440176,0.579664,0.0,1.0,1.0,0.0


Dtype route


,num__pclass,num__age,num__fare,cat__sex_female,cat__sex_male
0,0.844973,1.596599,-0.835759,0.0,1.0
1,-0.340934,-0.107641,-0.451979,1.0,0.0
2,0.844973,0.349594,-0.258225,0.0,1.0
3,-1.526842,-0.440176,0.579664,0.0,1.0
4,0.844973,-0.091014,0.551719,0.0,1.0
,num__pclass,num__age,num__fare,cat__sex_female,cat__sex_male
0,0.844973,1.596599,-0.835759,0.0,1.0
1,-0.340934,-0.107641,-0.451979,1.0,0.0
2,0.844973,0.349594,-0.258225,0.0,1.0
3,-1.526842,-0.440176,0.579664,0.0,1.0


## Randomized search

The dtype route also searches the gallery's imputer strategy, selector
percentile, and classifier regularization. Moxie wraps the column
transformer with `as_table_transformer()` and provides row selection
callbacks so every candidate fits its own preprocessing in each fold.
A typed parameter applier configures each nested step. Integral class
labels use Float64 throughout this pipeline so chi-squared selection
and logistic regression receive the same targets.

In [8]:
%%mojo randomized_search
from std.utils import Variant

from moxie.compose import ColumnEntry, Columns, ColumnTable, ColumnTransformer, make_column_selector
from moxie.feature_selection import BuiltinScorer, SelectPercentile
from moxie.impute import SimpleImputer
from moxie.linear_model import LogisticRegression
from moxie.model_selection import DenseSupervisedAdapter, ErrorPolicy, ParameterGrid, RandomizedSearchCV
from moxie.pipeline import Pipeline, make_pipeline
from moxie.preprocessing import OneHotEncoder, StandardScaler
from moxie_nb_io import load, save, save_floats, save_ints, save_strings
from numojo import Item, NDArray, Shape
from titanic_table import load_table

comptime Value = Variant[String, Float64]
comptime STRATEGY = "preprocessor__num__imputer__strategy"
comptime PERCENTILE = "preprocessor__cat__selector__percentile"
comptime REGULARIZATION = "classifier__C"


def main() raises:
    var train = load_table("train", False)
    var test = load_table("test", False)
    var y_train = load("y_train")
    var y_test = load("y_test")
    var category: List[String] = ["category"]
    var num = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    var cat = make_pipeline(
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        SelectPercentile(BuiltinScorer("chi2"), percentile=50),
    )
    var columns = ColumnTransformer(
        ColumnEntry("num", num^, Columns(make_column_selector(dtype_exclude=category.copy()))),
        ColumnEntry("cat", cat^, Columns(make_column_selector(dtype_include=category^))),
    )
    var model = Pipeline(
        columns^.as_table_transformer(), LogisticRegression[label_dtype=DType.float64](),
        transformer_name="preprocessor", estimator_name="classifier",
    )
    comptime Model = type_of(model)
    comptime Table = ColumnTable[]
    comptime Adapter = DenseSupervisedAdapter[Model, V=Value]

    def configure(mut estimator: Model, params: Dict[String, Value]) raises:
        var config = estimator.get_params()
        ref entries = config.transformer.transformers
        ref numeric = entries.head.transformer.estimator.value()
        ref categorical = entries.tail.head.transformer.estimator.value()
        _ = numeric.transformer.set_params(strategy=params[STRATEGY][String].copy())
        _ = categorical.estimator.set_params(percentile=params[PERCENTILE][Float64])
        config.estimator.C = params[REGULARIZATION][Float64]
        _ = estimator.set_params(config)

    var grid = Dict[String, List[Value]]()
    grid[STRATEGY] = [Value(String("mean")), Value(String("median"))]
    grid[PERCENTILE] = [Value(Float64(10)), Value(Float64(30)), Value(Float64(50)), Value(Float64(70))]
    grid[REGULARIZATION] = [Value(Float64(0.1)), Value(Float64(1)), Value(Float64(10)), Value(Float64(100))]
    var adapter = Adapter(
        configure, classifier=True,
        input_rows=Adapter.InputRows(Table.rows, Table.take_rows),
    )
    var search = RandomizedSearchCV[Adapter](
        model, ParameterGrid[Value](grid), adapter,
        n_iter=10, random_state=0, error_score=ErrorPolicy(True, 0),
    )
    _ = search.fit(train, y_train)

    var results = search.cv_results_()
    var strategies = List[String]()
    var percentiles = List[Float64]()
    var regularizations = List[Float64]()
    var n_splits = results.scores.test_scores.shape[1]
    var fold_scores = NDArray[DType.float64](Shape(len(results.params), n_splits))
    for candidate in range(len(results.params)):
        ref params = results.params[candidate]
        strategies.append(params[STRATEGY][String].copy())
        percentiles.append(params[PERCENTILE][Float64])
        regularizations.append(params[REGULARIZATION][Float64])
        for fold in range(n_splits):
            fold_scores[Item(candidate, fold)] = results.scores.test_scores.item(candidate, fold, 0)
    save_strings("strategy", strategies)
    save_floats("percentile", percentiles)
    save_floats("C", regularizations)
    save("fold_scores", fold_scores)
    save("mean_test_score", results.mean_test_score)
    save("std_test_score", results.std_test_score)
    save("rank_test_score", results.rank_test_score)
    save_ints("best_index", [search.best_index_()])
    save_floats("best_score", [search.best_score_()])
    save("prediction", search.predict(test))
    save_floats("accuracy", [search.score(test, y_test)])

In [9]:
search_sources = {
    "split": "sklearn/model_selection/_split.py",
    "validation": "sklearn/model_selection/_validation.py",
    "search": "sklearn/model_selection/_search.py",
    "indexing": "sklearn/utils/metaestimators.py",
}
verify_reference(search_sources.values())
indexing = load_pinned(search_sources["indexing"])
split = load_pinned(search_sources["split"], register=True)
validation = load_pinned(search_sources["validation"], register=True)
validation._safe_split = indexing._safe_split
search_module = load_pinned(search_sources["search"])
for method, key in [
    (split.StratifiedKFold._make_test_folds, "split"),
    (validation._fit_and_score, "validation"),
    (validation._safe_split, "indexing"),
    (search_module.RandomizedSearchCV.fit, "search"),
    (search_module.ParameterSampler.__iter__, "search"),
]:
    pinned_code(method, search_sources[key])
assert search_module._fit_and_score is validation._fit_and_score

strategy = "preprocessor__num__imputer__strategy"
percentile = "preprocessor__cat__selector__percentile"
regularization = "classifier__C"
param_grid = {
    strategy: ["mean", "median"],
    percentile: [10, 30, 50, 70],
    regularization: [0.1, 1.0, 10.0, 100.0],
}
search_columns = preprocessor(non_category_selector, category_selector)
search_model = pipeline.Pipeline([
    ("preprocessor", search_columns),
    ("classifier", LogisticRegression()),
])
sk_search = search_module.RandomizedSearchCV(
    search_model, param_grid, n_iter=10, random_state=0,
    cv=split.StratifiedKFold(), error_score="raise",
).fit(X_train, y_train)
sk_results = sk_search.cv_results_
mx_search = nb.results["randomized_search"]
assert mx_search["prediction"].dtype == np.float64

for key, field in [(strategy, "strategy"), (percentile, "percentile"), (regularization, "C")]:
    nb.compare(f"Search sampled {field}", [params[key] for params in sk_results["params"]],
               mx_search[field], exact=True)
nb.compare("Search fold accuracy", np.column_stack([
    sk_results[f"split{i}_test_score"] for i in range(sk_search.n_splits_)
]), mx_search["fold_scores"], exact=True)
# Moxie search arrays retain the metric axis for a single scorer.
for field in ("mean_test_score", "std_test_score"):
    nb.compare(f"Search {field}", sk_results[field][:, None], mx_search[field], rtol=0, atol=1e-15,
               reason="Float64 averaging can round differently.")
nb.compare("Search ranks", sk_results["rank_test_score"][:, None], mx_search["rank_test_score"], exact=True)
nb.compare("Search best index", [sk_search.best_index_], mx_search["best_index"], exact=True)
nb.compare("Search best CV score", [sk_search.best_score_], mx_search["best_score"],
           rtol=0, atol=1e-15, reason="Float64 averaging can round differently.")
nb.compare("Search refit predictions", sk_search.predict(X_test), mx_search["prediction"], exact=True)
nb.compare("Search held-out accuracy", [sk_search.score(X_test, y_test)],
           mx_search["accuracy"], exact=True)

display(pd.DataFrame(sk_results).sort_values("rank_test_score")[[
    "mean_test_score", "std_test_score", f"param_{strategy}",
    f"param_{percentile}", f"param_{regularization}",
]].head())
print(f"Moxie held-out accuracy after refit {mx_search['accuracy'][0]:.3f}")
nb.report()
nb.assert_parity()

,mean_test_score,std_test_score,param_preprocessor__num__imputer__strategy,param_preprocessor__cat__selector__percentile,param_classifier__C
3,0.789902,0.013078,mean,30,0.1
1,0.788959,0.018503,mean,70,10.0
0,0.788954,0.016755,median,30,1.0
2,0.788954,0.016755,mean,30,1.0
4,0.788954,0.016755,mean,10,10.0


Moxie held-out accuracy after refit 0.798


check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
Dtype route numeric selector,PASS,"(3,)","(3,)",,,exact,0 mismatches,
Dtype route categorical selector,PASS,"(2,)","(2,)",,,exact,0 mismatches,
Dtype route input schema,PASS,"(5,)","(5,)",,,exact,0 mismatches,
named feature names,PASS,"(6,)","(6,)",,,exact,0 mismatches,
named output_indices_,PASS,"(6,)","(6,)",,,exact,0 mismatches,
named train numeric output,PASS,"(1047, 2)","(1047, 2)",1.29e-14,2.25e-11,"rtol=1e-12, atol=1e-12",,Float64 scaler reductions can sum in different orders.
named train selected one-hot output,PASS,"(1047, 4)","(1047, 4)",,,exact,0 mismatches,
named test numeric output,PASS,"(262, 2)","(262, 2)",1.24e-14,1.24e-12,"rtol=1e-12, atol=1e-12",,Float64 scaler reductions can sum in different orders.
named test selected one-hot output,PASS,"(262, 4)","(262, 4)",,,exact,0 mismatches,
named classifier predictions,PASS,"(262,)","(262,)",,,exact,0 mismatches,
